In [1]:
import numpy as np
import torchvision as thv

from torch import torch
from torch.utils.data import DataLoader

# from common.utils import Utils
# from common.nn_model import NNModel, NNTrainParams
# from common.feed_fwd_nn import FeedFwdNN, FeedFwdNNParams

from common.nn.enum.loss import Loss
from common.nn.nn_model import NNModel
from common.nn.enum.device import Device
from common.nn.net.feed_fwd_nn import FeedFwdNN
from common.nn.params.nn_params import NNParams
from common.nn.dataset.nn_dataset import NNDataset
from common.nn.params.nn_train_params import NNTrainParams
from common.nn.params.nn_model_params import NNModelParams

In [2]:
DS_MEAN : float = 0.1307
DS_STD  : float = 0.3081

ds = NNDataset(
    ds_class=thv.datasets.MNIST
    , transform=thv.transforms.Compose(
        [
            thv.transforms.ToTensor()
            , thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD)
        ]
    )
)

print(
    ds.input_dim
    , ds.output_dim
    , len(ds.train_loader.dataset)
    , len(ds.val_loader.dataset)
    , len(ds.test_loader.dataset)
)

784 10 60000 1000 9000


In [4]:
model = NNModel(
    net=FeedFwdNN(
        params=NNParams(
            dropout_prob=0.5
            , hidden_dims=[]
            , input_dim=ds.input_dim
            , output_dim=ds.output_dim
        )
    )
)

In [29]:
model.train(
    params=NNTrainParams(
        n_epochs=100
        , val_loader=ds.val_loader
        , train_loader=ds.train_loader
    )
)

{'loss': cross_entropy, 'device': cpu, 'dims': [784, 10], 'dropout_prob': 0.5, 'activation_fn': leaky_relu, 'n_epochs': 100, 'optim_params': {'optim': adam, 'lr_start': 0.9, 'momentum': (0.9, 0.999), 'weight_decay': 0.0005}, 'scheduler_params': {'factor': 0.9, 'patience': 5, 'threshold': 0.0001}}


Training: 100%|██████████| 100/100 [04:56<00:00,  2.96s/it, error=0.0980, lr=0.2542]


('[device=cpu, loss=cross_entropy] x FeedFwdNN={dims=[784, 10], activation=leaky_relu, dropout=0.50} x Train={n_epochs=100, OptimParams=[optim=adam, lr_start=9e-01, weight_decay=5e-04, momentum=(0.9, 0.999)], SchedulerParams=[patience=5, factor=9e-01, threshold=1e-04]}',
 array([NNIterationDataPoint(iter_idx=0, epoch_idx=0, batch_idx=0, train_edp=NNEvaluationDataPoint(f1=0.6713666666666667, recall=0.6713666666666667, accuracy=0.6713666666666667, precision=0.6713666666666667, loss=52.12140655517578, error=0.32863333333333333), val_edp=NNEvaluationDataPoint(f1=0.658, recall=0.658, accuracy=0.658, precision=0.658, loss=77.77366638183594, error=0.34199999999999997)),
        NNIterationDataPoint(iter_idx=1, epoch_idx=1, batch_idx=0, train_edp=NNEvaluationDataPoint(f1=0.6644, recall=0.6644, accuracy=0.6644, precision=0.6644, loss=73.97647857666016, error=0.3356), val_edp=NNEvaluationDataPoint(f1=0.693, recall=0.693, accuracy=0.693, precision=0.693, loss=64.45458221435547, error=0.3070000000

In [4]:
n_epochs = 1
lrs = [0.001]
optims = ["adam"]

dropout_probs = [0.5]
hidden_dimss = [
    []
    # , [500]
    # , [1000]
    # , [500, 1000]
]

models = [
    model
        for model in [
            NNModel(
                device="cpu"
                , net=FeedFwdNN(
                    params=FeedFwdNNParams(
                        input_dim=28*28
                        , output_dim=10
                        , hidden_dims=hidden_dims
                        , dropout_prob=dropout_prob
                    )
                )
            )
                for dropout_prob in dropout_probs
                for hidden_dims in hidden_dimss
        ]
]

train_params = [
    train_param
        for train_param in [
            NNTrainParams(
                optim=optim
                , learning_rate=lr
                , n_epochs=n_epochs
                , val_loader=val_loader
                , train_loader=train_loader
            )
                for lr in lrs
                for optim in optims 
        ]
]

trains = {
    train_str: (model, train_idps)
        for model, (train_str, train_idps) in [
            (model, model.train(params=train_param))
                for model in models
                for train_param in train_params
        ]
}

FeedFwdNN{dims=[784, 10], dropout=0.5} x [epochs=1, lr=0.001, optim=adam]: 100%|██████████| 1/1 [00:05<00:00,  5.01s/it, error: 0.8223]


In [ ]:
top_model_names = [
    kvp[0] for kvp in sorted(
        list(trains.items())
        , key=lambda kvp: min(
            kvp[1][1]
            , key=lambda idp: idp.val_error
        ).val_error
    )[:5]
]

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=1
    , fig_size=(25, 20)
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Losses"
    , x=[idp.iter_idx for idp in trains[top_model_names[0]][1]]
    , yss_legend=[[f"{loss_type} of {model_name}" for loss_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.train_loss for model_idp in trains[model_name][1]], [model_idp.val_loss for model_idp in trains[model_name][1]]] for model_name in top_model_names]
)

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=1
    , fig_size=(25, 20)
    , y_axis_label="Error"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Errors"
    , x=[idp.iter_idx for idp in trains[top_model_names[0]][1]]
    , yss_legend=[[f"{loss_type} of {model_name}" for loss_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.train_error for model_idp in trains[model_name][1]], [model_idp.val_error for model_idp in trains[model_name][1]]] for model_name in top_model_names]
)

In [ ]:
print(f"best model is {top_model_names[0]} which achieves validation error of {min(trains[top_model_names[0]][1], key=lambda idp: idp.val_error).val_error:.4f}")